# Leukemia Detection — EfficientNetB0 Method 4: ML Classifier

**Approach:** Use frozen EfficientNetB0 as feature extractor. Extract features from all images,
then train traditional ML classifiers (SVM + Random Forest) on extracted features.
No data augmentation. Models saved with pickle.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, pickle
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

## Data Loading

In [ ]:
# Dataset paths
train_dir = '/kaggle/input/datasets/avk256/cnmc-leukemia/fold_0'
val_dir = '/kaggle/input/datasets/avk256/cnmc-leukemia/fold_1'
test_dir = '/kaggle/input/datasets/avk256/cnmc-leukemia/fold_2'

for name, path in [('Train', train_dir), ('Validation', val_dir), ('Test', test_dir)]:
    if os.path.exists(path):
        print(f'\n{name} set:')
        for cls in sorted(os.listdir(path)):
            cls_path = os.path.join(path, cls)
            if os.path.isdir(cls_path):
                print(f'  {cls}: {len(os.listdir(cls_path))} images')

## Data Generators (No Augmentation)

In [ ]:
# Data generators — rescaling ONLY, no augmentation
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

datagen = ImageDataGenerator(rescale=1.0/255)

train_gen = datagen.flow_from_directory(train_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=True)
val_gen = datagen.flow_from_directory(val_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=False)
test_gen = datagen.flow_from_directory(test_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', shuffle=False)

print(f'Class mapping: {train_gen.class_indices}')
print(f'Training: {train_gen.samples} | Validation: {val_gen.samples} | Test: {test_gen.samples}')

## Feature Extractor (EfficientNetB0)

In [ ]:
# Build feature extractor (frozen pretrained model)
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

feature_extractor = Sequential([base_model, GlobalAveragePooling2D()])
feature_extractor.summary()

## Extract Features

In [ ]:
# Extract features from all images
print('Extracting training features...')
train_features = feature_extractor.predict(train_gen, verbose=1)
train_labels = train_gen.classes

print('Extracting validation features...')
val_features = feature_extractor.predict(val_gen, verbose=1)
val_labels = val_gen.classes

print('Extracting test features...')
test_features = feature_extractor.predict(test_gen, verbose=1)
test_labels = test_gen.classes

# Scale features
scaler = StandardScaler()
train_features_scaled = scaler.fit_transform(train_features)
val_features_scaled = scaler.transform(val_features)
test_features_scaled = scaler.transform(test_features)

print(f'\nFeature shape: {train_features.shape}')
print(f'Training: {len(train_labels)} | Validation: {len(val_labels)} | Test: {len(test_labels)}')

## Train ML Classifiers (SVM + Random Forest)

In [ ]:
# Train SVM classifier
print('Training SVM...')
svm = SVC(kernel='rbf', C=10, gamma='scale', probability=True)
svm.fit(train_features_scaled, train_labels)
svm_val_acc = svm.score(val_features_scaled, val_labels)
svm_test_acc = svm.score(test_features_scaled, test_labels)
print(f'SVM — Val Accuracy: {svm_val_acc*100:.2f}% | Test Accuracy: {svm_test_acc*100:.2f}%')

# Train Random Forest classifier
print('\nTraining Random Forest...')
rf = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42, n_jobs=-1)
rf.fit(train_features_scaled, train_labels)
rf_val_acc = rf.score(val_features_scaled, val_labels)
rf_test_acc = rf.score(test_features_scaled, test_labels)
print(f'RF  — Val Accuracy: {rf_val_acc*100:.2f}% | Test Accuracy: {rf_test_acc*100:.2f}%')

# Save models with pickle
with open('/kaggle/working/efficientnetb0_ml_models.pkl', 'wb') as f:
    pickle.dump({'svm': svm, 'rf': rf, 'scaler': scaler}, f)
print('\nML models saved with pickle.')

## Evaluation

In [ ]:
# Evaluate best classifier on test set
svm_preds = svm.predict(test_features_scaled)
rf_preds = rf.predict(test_features_scaled)
class_names = list(test_gen.class_indices.keys())

# Choose best model
best_name, best_preds, best_acc = ('SVM', svm_preds, svm_test_acc) if svm_test_acc >= rf_test_acc else ('Random Forest', rf_preds, rf_test_acc)
print(f'Best Classifier: {best_name} ({best_acc*100:.2f}%)')

# Confusion Matrix
cm = confusion_matrix(test_labels, best_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=class_names, yticklabels=class_names, annot_kws={'size': 16})
plt.title('Confusion Matrix — EfficientNetB0 ({best_name})', fontsize=16, fontweight='bold')
plt.xlabel('Predicted'); plt.ylabel('True'); plt.tight_layout(); plt.show()

print(f'\n=== {best_name} Classification Report ===')
print(classification_report(test_labels, best_preds, target_names=class_names))

# Also show the other classifier
other_name, other_preds = ('Random Forest', rf_preds) if best_name == 'SVM' else ('SVM', svm_preds)
print(f'\n=== {other_name} Classification Report ===')
print(classification_report(test_labels, other_preds, target_names=class_names))

## Prediction Visualization

In [ ]:
# Visualize predictions using best ML classifier
test_gen.reset()
images, labels = next(test_gen)

# Get features for this batch and predict
batch_features = feature_extractor.predict(images)
batch_features_scaled = scaler.transform(batch_features)
if svm_test_acc >= rf_test_acc:
    batch_preds = svm.predict(batch_features_scaled)
else:
    batch_preds = rf.predict(batch_features_scaled)

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i])
    true_label = class_names[int(labels[i])]
    pred_label = class_names[int(batch_preds[i])]
    color = 'green' if true_label == pred_label else 'red'
    ax.set_title(f'True: {true_label}\nPred: {pred_label}', fontsize=10, color=color, fontweight='bold')
    ax.axis('off')
plt.suptitle('EfficientNetB0 — ML Classifier Predictions', fontsize=16, fontweight='bold')
plt.tight_layout(); plt.show()